# Scraping Lesiones y Valores de Mercado — TFG Guillermo Jiménez

Enriquece el dataset con datos de Transfermarkt:

1. **Historial de valor de mercado** por jugador (tabla MV History)
2. **Lesiones por temporada** — días perdidos, tendencias, métricas derivadas
3. **Ingresos de club** (`club_revenue_eur`, `revenue_tier`) desde diccionario Deloitte

**Input:** `Jugadores_Combinados.xlsx`  
**Output:** `Dataset_Definitivo.xlsx`

In [ ]:
!pip install -q requests beautifulsoup4 lxml openpyxl pandas scipy numpy

## Imports y configuración

In [ ]:
import requests
import time
import random
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

## Rutas y parámetros

In [ ]:
import os
DB = '/content'

INPUT_PATH  = os.path.join(DB, 'Jugadores_Combinados.xlsx')
OUTPUT_PATH = os.path.join(DB, 'Dataset_Definitivo.xlsx')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'de-DE,de;q=0.9,en-US;q=0.8',
    'Referer': 'https://www.transfermarkt.com/',
}

MIN_DELAY = 3.0
MAX_DELAY = 6.0

SEASONS = [2023, 2022, 2021, 2020, 2019]

## Ingresos de club por equipo

Fuente: Deloitte Football Money League 2024 (datos temporada 2022-23).  
`revenue_tier`: `'elite'` (>400M€), `'top'` (200-400M€), `'mid'` (100-200M€), `'small'` (<100M€)

In [ ]:
CLUB_REVENUES = {
    # Premier League
    'Manchester City':   (831, 'elite'),
    'Manchester Utd':    (749, 'elite'),
    'Liverpool':         (703, 'elite'),
    'Arsenal':           (551, 'elite'),
    'Chelsea':           (512, 'elite'),
    'Tottenham':         (450, 'elite'),
    'Newcastle Utd':     (260, 'top'),
    'West Ham':          (218, 'top'),
    'Aston Villa':       (206, 'top'),
    'Brighton':          (194, 'top'),
    'Everton':           (170, 'mid'),
    'Crystal Palace':    (145, 'mid'),
    'Fulham':            (130, 'mid'),
    'Wolves':            (128, 'mid'),
    'Brentford':         (120, 'mid'),
    "Nott'ham Forest":   (118, 'mid'),
    'Bournemouth':       (115, 'mid'),
    'Burnley':           ( 92, 'small'),
    'Sheffield Utd':     ( 78, 'small'),
    'Luton Town':        ( 55, 'small'),
    # LaLiga
    'Real Madrid':       (831, 'elite'),
    'Barcelona':         (800, 'elite'),
    'Atlético Madrid':   (450, 'elite'),
    'Athletic Club':     (185, 'mid'),
    'Real Sociedad':     (168, 'mid'),
    'Betis':             (155, 'mid'),
    'Sevilla':           (148, 'mid'),
    'Villarreal':        (142, 'mid'),
    'Valencia':          (115, 'mid'),
    'Osasuna':           ( 80, 'small'),
    'Rayo Vallecano':    ( 60, 'small'),
    'Getafe':            ( 58, 'small'),
    'Celta Vigo':        ( 70, 'small'),
    'Mallorca':          ( 55, 'small'),
    'Girona':            ( 52, 'small'),
    'Las Palmas':        ( 40, 'small'),
    'Alavés':            ( 38, 'small'),
    'Espanyol':          ( 65, 'small'),
    'Cádiz':             ( 35, 'small'),
    'Granada':           ( 33, 'small'),
    # Serie A
    'Inter':             (440, 'elite'),
    'Milan':             (415, 'elite'),
    'Juventus':          (395, 'top'),
    'Roma':              (245, 'top'),
    'Lazio':             (185, 'mid'),
    'Napoli':            (280, 'top'),
    'Atalanta':          (175, 'mid'),
    'Fiorentina':        (135, 'mid'),
    'Bologna':           (110, 'mid'),
    'Torino':            ( 90, 'small'),
    'Udinese':           ( 70, 'small'),
    'Genoa':             ( 65, 'small'),
    'Lecce':             ( 55, 'small'),
    'Cagliari':          ( 50, 'small'),
    'Verona':            ( 60, 'small'),
    'Hellas Verona':     ( 60, 'small'),
    'Frosinone':         ( 40, 'small'),
    'Sassuolo':          ( 75, 'small'),
    'Salernitana':       ( 38, 'small'),
    'Como':              ( 35, 'small'),
    'Monza':             ( 80, 'small'),
    'Empoli':            ( 48, 'small'),
    'Parma':             ( 42, 'small'),
    # Bundesliga
    'Bayern Munich':     (854, 'elite'),
    'Dortmund':          (480, 'elite'),
    'RB Leipzig':        (310, 'top'),
    'Leverkusen':        (250, 'top'),
    'Eint Frankfurt':    (210, 'top'),
    'Stuttgart':         (185, 'mid'),
    'Hoffenheim':        (165, 'mid'),
    'Wolfsburg':         (160, 'mid'),
    'Freiburg':          (140, 'mid'),
    'Gladbach':          (195, 'mid'),
    'Union Berlin':      (130, 'mid'),
    'Werder Bremen':     (125, 'mid'),
    'Augsburg':          ( 95, 'small'),
    'Mainz 05':          ( 90, 'small'),
    'Köln':              ( 88, 'small'),
    'Heidenheim':        ( 45, 'small'),
    'St. Pauli':         ( 55, 'small'),
    'Hamburger SV':      ( 70, 'small'),
    # Ligue 1
    'Paris S-G':         (800, 'elite'),
    'Lyon':              (255, 'top'),
    'Marseille':         (235, 'top'),
    'Monaco':            (210, 'top'),
    'Lens':              (145, 'mid'),
    'Lille':             (140, 'mid'),
    'Nice':              (135, 'mid'),
    'Rennes':            (128, 'mid'),
    'Strasbourg':        ( 90, 'small'),
    'Nantes':            ( 85, 'small'),
    'Reims':             ( 75, 'small'),
    'Toulouse':          ( 70, 'small'),
    'Brest':             ( 65, 'small'),
    'Montpellier':       ( 60, 'small'),
    'Le Havre':          ( 48, 'small'),
    'Lorient':           ( 50, 'small'),
    'Metz':              ( 42, 'small'),
    'Clermont Foot':     ( 38, 'small'),
    'Auxerre':           ( 40, 'small'),
    'Angers':            ( 36, 'small'),
}

## Funciones de scraping: historial MV y lesiones

In [ ]:
def scrape_mv_history(tm_player_id, session):
    """
    Obtiene el historial de valores de mercado de un jugador desde Transfermarkt.
    Devuelve lista de dicts: [{date, market_value_eur, club_at_time}, ...]
    """
    url = f'https://www.transfermarkt.com/x/marktwertverlauf/spieler/{tm_player_id}'

    try:
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        resp = session.get(url, headers=HEADERS, timeout=20)

        if resp.status_code != 200:
            return []

        soup = BeautifulSoup(resp.text, 'lxml')

        scripts = soup.find_all('script')
        json_data = None
        for script in scripts:
            text = script.string or ''
            if 'marktwertverlauf' in text and 'data:' in text:
                match = re.search(r'"series":\s*\[.*?"data":\s*(\[.*?\])', text, re.DOTALL)
                if match:
                    import json
                    try:
                        json_data = json.loads(match.group(1))
                    except json.JSONDecodeError:
                        pass
                break

        if not json_data:
            return []

        records = []
        for point in json_data:
            try:
                date = point.get('datum', '')
                value = point.get('y', 0)
                club  = point.get('verein', '')
                records.append({
                    'date': date,
                    'market_value_eur': int(value) if value else None,
                    'club_at_time': club,
                })
            except (TypeError, ValueError):
                continue

        return records

    except Exception:
        return []


def scrape_injury_history(tm_player_id, session):
    """
    Obtiene el historial de lesiones de un jugador desde Transfermarkt.
    Devuelve: (records, by_season)
    """
    url = f'https://www.transfermarkt.com/x/verletzungen/spieler/{tm_player_id}'

    try:
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        resp = session.get(url, headers=HEADERS, timeout=20)

        if resp.status_code != 200:
            return [], {}

        soup = BeautifulSoup(resp.text, 'lxml')

        table = soup.find('table', class_='items')
        if not table:
            return [], {}

        rows = table.find_all('tr', class_=['odd', 'even'])
        records = []
        by_season = {}

        for row in rows:
            cells = row.find_all('td')
            if len(cells) < 6:
                continue

            season_raw = cells[0].get_text(strip=True)
            injury_type = cells[1].get_text(strip=True)
            date_from   = cells[2].get_text(strip=True)
            date_until  = cells[3].get_text(strip=True)
            days_raw    = cells[4].get_text(strip=True)
            games_raw   = cells[5].get_text(strip=True)

            season_year = None
            season_match = re.match(r'(\d{2})/\d{2}', season_raw)
            if season_match:
                yy = int(season_match.group(1))
                season_year = 2000 + yy

            days_match = re.search(r'(\d+)', days_raw)
            days = int(days_match.group(1)) if days_match else 0

            records.append({
                'season': season_raw,
                'season_year': season_year,
                'injury_type': injury_type,
                'date_from': date_from,
                'date_until': date_until,
                'days_missed': days,
            })

            if season_year:
                by_season[season_year] = by_season.get(season_year, 0) + days

        return records, by_season

    except Exception:
        return [], {}

## Función: calcular métricas derivadas de lesiones

In [ ]:
def compute_injury_metrics(by_season, all_records):
    """
    Calcula todas las métricas de lesión para un jugador dado su historial.
    """
    if not by_season:
        return {
            'injury_count': 0,
            'inj_days_last1': 0,
            'inj_days_avg_last2': 0.0,
            'inj_days_avg_last3': 0.0,
            'injury_days_per_season': 0.0,
            'injury_frequency': 0.0,
            'inj_trend_slope': 0.0,
            'inj_trend_pvalue': 1.0,
            'inj_trend_sig': False,
            'inj_ewa_days': 0.0,
            'inj_risk': 'low',
            'inj_series_type': 'stable',
        }

    sorted_seasons = sorted(by_season.items())
    years  = [s[0] for s in sorted_seasons]
    days   = [s[1] for s in sorted_seasons]

    injury_count = len(all_records)

    inj_last1 = days[-1] if len(days) >= 1 else 0
    inj_avg2  = np.mean(days[-2:]) if len(days) >= 2 else float(days[-1])
    inj_avg3  = np.mean(days[-3:]) if len(days) >= 3 else np.mean(days)

    inj_per_season = np.mean(days)
    inj_frequency  = injury_count / len(sorted_seasons) if sorted_seasons else 0

    if len(days) >= 3:
        slope, intercept, r, pvalue, stderr = stats.linregress(years, days)
    else:
        slope, pvalue = 0.0, 1.0

    inj_sig = pvalue < 0.05

    alpha = 0.4
    ewa = days[0]
    for d in days[1:]:
        ewa = alpha * d + (1 - alpha) * ewa

    if inj_per_season < 10:
        risk = 'low'
    elif inj_per_season < 40:
        risk = 'medium'
    else:
        risk = 'high'

    if abs(slope) < 2:
        series_type = 'stable'
    elif slope > 0:
        series_type = 'worsening'
    else:
        series_type = 'improving'

    return {
        'injury_count': injury_count,
        'inj_days_last1': round(inj_last1, 1),
        'inj_days_avg_last2': round(inj_avg2, 1),
        'inj_days_avg_last3': round(inj_avg3, 1),
        'injury_days_per_season': round(inj_per_season, 1),
        'injury_frequency': round(inj_frequency, 2),
        'inj_trend_slope': round(slope, 3),
        'inj_trend_pvalue': round(pvalue, 4),
        'inj_trend_sig': inj_sig,
        'inj_ewa_days': round(ewa, 1),
        'inj_risk': risk,
        'inj_series_type': series_type,
    }

## Ejecución principal: carga y scraping

In [23]:
print('=' * 60)
print('  TFG — Scraping Lesiones + Historial MV (Transfermarkt)')
print('=' * 60)

print('\nCargando Excel enriquecido...')

combined = pd.read_excel(INPUT_PATH)
print(f'  {len(combined)} jugadores en total')
print(f'  Columnas: {list(combined.columns)}')

# --- Normalizar nombres de columnas (soporte inglés y español) ---
COL_MAP = {
    # inglés → estándar interno
    'Player':        'Player',
    'Squad':         'Squad',
    'tm_player_id':  'tm_player_id',
    'League':        'League',
    # español → estándar interno
    'Jugador':                        'Player',
    'Equipo':                         'Squad',
    'ID del jugador en Transfermarkt':'tm_player_id',
    'Liga':                           'League',
}

rename = {col: std for col, std in COL_MAP.items() if col in combined.columns and col != std}
if rename:
    combined = combined.rename(columns=rename)
    print(f'  Columnas renombradas: {rename}')

# Verificar columna clave
if 'tm_player_id' not in combined.columns:
    raise ValueError(
        f"No se encontró la columna de ID de Transfermarkt.\n"
        f"Columnas disponibles: {list(combined.columns)}"
    )

# --- Reconstruir all_sheets y sheet_names ---
LEAGUE_TO_SHEET = {
    'Premier League': 'PL Players',
    'LaLiga':         'LaLiga Players',
    'Serie A':        'Serie A Players',
    'Bundesliga':     'Bundesliga Players',
    'Ligue 1':        'Ligue 1 Players',
}

if 'League' in combined.columns:
    sheet_names = list(LEAGUE_TO_SHEET.values())
    all_sheets = {}
    for league, sheet in LEAGUE_TO_SHEET.items():
        mask = combined['League'] == league
        all_sheets[sheet] = combined[mask].copy().reset_index(drop=True)
    print('  Dividido por columna League')
elif '_sheet' in combined.columns:
    SHEET_TO_LEAGUE = {v: k for k, v in LEAGUE_TO_SHEET.items()}
    combined['League'] = combined['_sheet'].map(SHEET_TO_LEAGUE)
    sheet_names = list(LEAGUE_TO_SHEET.values())
    all_sheets = {}
    for sheet in sheet_names:
        mask = combined['_sheet'] == sheet
        all_sheets[sheet] = combined[mask].copy().reset_index(drop=True)
    print('  Dividido por columna _sheet')
else:
    print('  AVISO: sin columna League — usando hoja única')
    combined['League'] = 'Unknown'
    sheet_names = ['All Players']
    all_sheets = {'All Players': combined.copy()}

has_id = combined['tm_player_id'].notna()
print(f'  {has_id.sum()} jugadores con ID de Transfermarkt')

session = requests.Session()
session.trust_env = False

mv_history_rows    = []
injuries_by_season = []
injury_metrics_map = {}

player_ids = combined.loc[has_id, 'tm_player_id'].unique()
n = len(player_ids)
print(f'\nProcesando {n} jugadores...')

for i, tm_id in enumerate(player_ids):
    tm_id = int(tm_id)
    player_row = combined[combined['tm_player_id'] == tm_id].iloc[0]
    player_name = player_row.get('Player', f'ID_{tm_id}')

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{n}] {player_name}...')

    mv_records = scrape_mv_history(tm_id, session)
    for rec in mv_records:
        mv_history_rows.append({
            'tm_player_id': tm_id,
            'player_name': player_name,
            'squad': player_row.get('Squad', ''),
            **rec
        })

    inj_records, inj_by_season = scrape_injury_history(tm_id, session)

    for rec in inj_records:
        injuries_by_season.append({
            'tm_id': tm_id,
            'player': player_name,
            'squad': player_row.get('Squad', ''),
            **rec
        })

    metrics = compute_injury_metrics(inj_by_season, inj_records)
    injury_metrics_map[tm_id] = metrics

    if (i + 1) % 100 == 0:
        pd.DataFrame(mv_history_rows).to_csv('partial_mv_history.csv', index=False)
        pd.DataFrame(injuries_by_season).to_csv('partial_injuries.csv', index=False)
        print(f'  Progreso guardado ({i+1}/{n})')

  TFG — Scraping Lesiones + Historial MV (Transfermarkt)

Cargando Excel enriquecido...
  1808 jugadores en total
  Columnas: ['Liga', 'Jugador', 'Nacionalidad', 'Posición', 'Equipo', 'Edad', 'Año de nacimiento', 'Partidos jugados esta temporada', 'Titularidades', 'Minutos jugados esta temporada', 'Minutos totales/90', 'Goles', 'Asistencias', 'G+A', 'G+A sin contar penaltis', 'Goles que no son de penalti', 'Goles de penalti', 'Penaltis tirados', 'Amarillas', 'Rojas', 'Goles esperados', 'Goles esperados sin contar penaltis', 'Asistencias esperadas', 'Contribuciones de gol esperadas sin penaltis', 'Conducción de balón de al menos 10 metros hacia la portería rival', 'Pases progresivos *', 'Recepciones de pases progresivos', 'Goles/ 90 mins', 'Asistencias/ 90mins', 'G+A/ 90 mins', 'Goles sin contar penaltis/ 90 mins', 'G+A sin contar penaltis/ 90 mins', 'Goles esperados/ 90 mins', 'Asistencias esperadas/ 90 mins', '(Goles+Asistencias) esperados/ 90 mins', 'Goles esperados sin contar penalt

KeyboardInterrupt: 

## Enriquecimiento de hojas con métricas de lesión e ingresos

In [24]:
print('\nAñadiendo métricas a las hojas de jugadores...')

INJURY_COLS = [
    'injury_count', 'inj_days_last1', 'inj_days_avg_last2', 'inj_days_avg_last3',
    'injury_days_per_season', 'injury_frequency',
    'inj_trend_slope', 'inj_trend_pvalue', 'inj_trend_sig',
    'inj_ewa_days', 'inj_risk', 'inj_series_type',
]

for col in INJURY_COLS:
    default = 0 if col not in ('inj_risk', 'inj_series_type', 'inj_trend_sig') else \
              ('low' if 'risk' in col else ('stable' if 'type' in col else False))
    for sheet in sheet_names:
        all_sheets[sheet][col] = default

for sheet in sheet_names:
    df = all_sheets[sheet]
    id_col = 'tm_player_id' if 'tm_player_id' in df.columns else None
    if id_col is None:
        continue

    for idx, row in df.iterrows():
        tm_id = row.get(id_col)
        if pd.isna(tm_id):
            continue
        tm_id = int(tm_id)
        if tm_id in injury_metrics_map:
            for col, val in injury_metrics_map[tm_id].items():
                df.at[idx, col] = val

    squad_col = 'Squad' if 'Squad' in df.columns else 'Equipo'
    df['club_revenue_eur'] = df[squad_col].map(
        lambda s: CLUB_REVENUES.get(str(s), (None, None))[0]
    )
    df['revenue_tier'] = df[squad_col].map(
        lambda s: CLUB_REVENUES.get(str(s), (None, None))[1]
    )

    all_sheets[sheet] = df

print('Calculando años de contrato restantes...')

for sheet in sheet_names:
    df = all_sheets[sheet]
    if 'contract_until' not in df.columns:
        df['contract_years_remaining'] = None
        continue

    def calc_years(val):
        if pd.isna(val):
            return None
        try:
            import re as _re
            match = _re.search(r'(\d{4})', str(val))
            if match:
                year = int(match.group(1))
                return max(0, year - 2024)
        except Exception:
            pass
        return None

    df['contract_years_remaining'] = df['contract_until'].apply(calc_years)
    all_sheets[sheet] = df


Añadiendo métricas a las hojas de jugadores...
Calculando años de contrato restantes...


## Guardar Excel final

In [25]:
print(f'\nGuardando "{OUTPUT_PATH}"...')

mv_df  = pd.DataFrame(mv_history_rows)
inj_df = pd.DataFrame(injuries_by_season)

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet in sheet_names:
        all_sheets[sheet].to_excel(writer, sheet_name=sheet, index=False)
        print(f'  {sheet}')

    if not mv_df.empty:
        mv_df.to_excel(writer, sheet_name='MV History', index=False)
        print(f'  MV History ({len(mv_df)} filas)')

    if not inj_df.empty:
        inj_df.to_excel(writer, sheet_name='Injuries by Season', index=False)
        print(f'  Injuries by Season ({len(inj_df)} filas)')

print(f'\nGuardado: {OUTPUT_PATH}')
print(f'   Jugadores procesados: {len(player_ids)}')
print(f'   Puntos de historial MV: {len(mv_history_rows)}')
print(f'   Registros de lesión: {len(injuries_by_season)}')


Guardando "/content/Dataset_Definitivo.xlsx"...
  PL Players
  LaLiga Players
  Serie A Players
  Bundesliga Players
  Ligue 1 Players
  Injuries by Season (835 filas)

Guardado: /content/Dataset_Definitivo.xlsx
   Jugadores procesados: 1736
   Puntos de historial MV: 0
   Registros de lesión: 835
